# SPY · Daily EMA rejection benchmark
This is an analyst-defined proxy for a discretionary bearish rejection. It uses the same 25 shuffled 2020–2024 rebound episodes as the personal blind replay. The benchmark does not record choices in the personal review.

Inspect the trigger close and up to five later closes. Confirm the first red candle that closes below the previous close while EMA8 is below EMA21. For at least one of those EMAs, the previous close must be at or below the previous EMA, the current high must reach the current EMA, and the current close must finish below it. Otherwise record no setup after the sixth close. Lock decisions before scoring.

A success means SPY closed strictly lower five trading sessions after confirmation. Compare with five sessions after each initial rebound. A second comparison uses the same elapsed session in other episodes still unconfirmed at that close. Those controls may confirm later. The control rate gives each matched signal equal weight and can reuse episodes. A five-session outcome extending past 2024 is unavailable. All rates describe this selected historical sample.

In [ ]:
# @title Load the original review episodes
# @title Load historical SPY episodes
from pathlib import Path
import random
import pandas as pd

SOURCE_COMMIT = "0d0a3cfe5acb9410382b5e58a855270620f46078"
DATA = f"https://raw.githubusercontent.com/jamelof23/SPY-EMA-V2/{SOURCE_COMMIT}/data"
price_columns = ["Open", "High", "Low", "Close", "Volume"]
def load_episodes(data):
    bars = pd.read_csv(f"{data}/source/SPY_daily_2019_2025.csv",
                       usecols=["Date", *price_columns]).set_index("Date").sort_index()
    bars = bars.loc["2019-01-01":"2024-12-31"].copy()
    assert bars.index.is_unique and bars.index[-1] == "2024-12-31"
    for span in (8, 21):
        bars[f"EMA{span}"] = bars.Close.ewm(span=span, adjust=False).mean()
    flags = pd.read_csv(f"{data}/labels/historical_audit_through_2025.csv",
                        usecols=["asof_date", "split", "bear_pullback_asof"])
    tagged = flags.loc[(flags.split == "train") & (flags.bear_pullback_asof == 1), "asof_date"]
    positions = sorted(bars.index.get_loc(day) for day in tagged)
    starts = [pos for i, pos in enumerate(positions) if i == 0 or pos - positions[i-1] > 5]
    assert len(starts) == 25 and all(41 <= p < len(bars) - 5 for p in starts)
    random.Random(42).shuffle(starts)
    return bars, starts

bars, starts = load_episodes(DATA)


In [ ]:
# @title Record the first qualifying close and lock all 25 decisions
"""Evaluate a fixed definition of rejection on the existing 25 blind-review episodes."""
from pathlib import Path
import hashlib
import io
import json
import pandas as pd
import numpy as np

HORIZON = 5
LAST_REVIEW_OFFSET = 5

def rejected(previous, current):
    if not (current["Close"] < current["Open"] and
            current["Close"] < previous["Close"] and current["EMA8"] < current["EMA21"]):
        return ""
    levels=[]
    for name in ("EMA8", "EMA21"):
        if (previous["Close"] <= previous[name] and
            current["High"] >= current[name] and current["Close"] < current[name]):
            levels.append(name)
    return "+".join(levels)

def lock_decisions(bars, starts):
    records=[]
    for episode,start in enumerate(starts,1):
        observations=[]
        for offset in range(LAST_REVIEW_OFFSET + 1):
            day=start+offset
            # These are the only two rows available to the decision function.
            level=rejected(bars.iloc[day-1],bars.iloc[day])
            observations.append("R" if level else ("N" if offset==LAST_REVIEW_OFFSET else "W"))
            if level: break
        records.append(dict(episode=episode,start=int(start),confirmed=bool(level),
                            offset=offset,level=level,actions="".join(observations)))
    return records

def outcome(bars,day):
    if day+HORIZON >= len(bars): return None
    return float(bars.Close.iloc[day+HORIZON]/bars.Close.iloc[day]-1)

def benchmark(bars,decisions):
    rows=[]
    pool=[]
    for d in decisions:
        start,offset=d["start"],d["offset"]
        day=start+offset
        trigger_return=outcome(bars,start)
        signal_return=outcome(bars,day) if d["confirmed"] else None
        peers=[]
        if signal_return is not None:
            for other in decisions:
                if other["episode"]==d["episode"]: continue
                if other["confirmed"] and other["offset"]<=offset: continue
                value=outcome(bars,other["start"]+offset)
                if value is None: continue
                peers.append(value)
                pool.append(dict(signal_episode=d["episode"],control_episode=other["episode"],
                                 offset=offset,control_return=value))
        rows.append(dict(episode=d["episode"],trigger_date=bars.index[start],actions=d["actions"],
                         level=d["level"],confirmed=d["confirmed"],offset=offset,
                         decision_date=bars.index[day],trigger_return=trigger_return,
                         signal_return=signal_return,matched_control_count=len(peers),
                         matched_control_down_rate=np.mean(np.asarray(peers)<0) if peers else np.nan))
    return pd.DataFrame(rows), pd.DataFrame(pool)

records=lock_decisions(bars,starts)
assert len(records)==len(starts)==25
serialized=json.dumps(records,sort_keys=True,separators=(",",":")).encode()
folder=Path("spy_ema_rejection_benchmark")
folder.mkdir(exist_ok=True)
(folder/"fixed_decisions.json").write_bytes(serialized)
fingerprint=hashlib.sha256(serialized).hexdigest()
print("25 decisions locked before five-session scoring. SHA-256",fingerprint,flush=True)


In [ ]:
# @title Score the locked decisions and timing-matched controls
results,matched=benchmark(bars,records)
assert results.trigger_return.notna().all()
assert all(d<"2025" for d in results.decision_date)
scored=results[results.signal_return.notna()]
paired=scored[scored.matched_control_down_rate.notna()]
signal_down=int(scored.signal_return.lt(0).sum())
trigger_down=int(results.trigger_return.lt(0).sum())
print(f"Confirmed {results.confirmed.sum()} of {len(results)} | no setup {(~results.confirmed).sum()} | confirmed with outcomes beyond 2024 {results.confirmed.sum()-len(scored)}")
print(f"After confirmation: DOWN {signal_down}/{len(scored)} ({signal_down/len(scored):.1%}) | median SPY return {scored.signal_return.median():+.2%}")
print(f"After any initial rebound: DOWN {trigger_down}/{len(results)} ({trigger_down/len(results):.1%}) | median SPY return {results.trigger_return.median():+.2%}")
print(f"After the initial rebound in these same confirmed/scorable episodes: DOWN {int(scored.trigger_return.lt(0).sum())}/{len(scored)}")
if len(paired):
    control_rate=float(paired.matched_control_down_rate.mean())
    signal_rate=float(paired.signal_return.lt(0).mean())
    print(f"Same elapsed session, other rebounds still unconfirmed: {control_rate:.1%} DOWN | confirmed: {signal_rate:.1%} DOWN | difference {(signal_rate-control_rate)*100:+.1f} percentage points | {len(paired)} matched signals")
print("Scoring ends December 2024. Episode-level results are saved separately to keep the personal replay blind.")
results.to_csv(folder/"episode_results.csv",index=False)
matched.to_csv(folder/"matched_controls.csv",index=False)

from IPython.display import HTML, display
from zipfile import ZipFile, ZIP_DEFLATED
import base64
stream=io.BytesIO()
with ZipFile(stream,"w",compression=ZIP_DEFLATED) as bundle:
    for name in ("fixed_decisions.json","episode_results.csv","matched_controls.csv"):
        bundle.write(folder/name,name)
data=base64.b64encode(stream.getvalue()).decode()
display(HTML(f'<a download="SPY_EMA_rejection_benchmark.zip" href="data:application/zip;base64,{data}">Download the rule decisions and comparisons (contains episode dates and outcomes)</a>'))


In [ ]:
# @title Compare your completed blind review (optional)
review_code = "" # @param {type:"string"}
def compare_personal(code):
    if not code.startswith("SPY25v1-"):
        raise ValueError("Use the resume code from the personal blind review.")
    records=[]
    offset=0
    for action in code[len("SPY25v1-"):]:
        if len(records)>=len(starts) or action not in "WRN":
            raise ValueError("Invalid or excessive decisions.")
        if action=="W":
            offset+=1
            if offset>LAST_REVIEW_OFFSET: raise ValueError("Too many waits.")
        else:
            i=len(records)
            records.append(dict(episode=i+1,start=starts[i],offset=offset,level="personal",
                                confirmed=(action=="R"),actions="W"*offset+action))
            offset=0
    if len(records)!=len(starts):
        print(f"{len(records)} of {len(starts)} episodes complete. Outcomes stay hidden.")
        return
    personal,_=benchmark(bars,records)
    scored=personal[personal.signal_return.notna()]
    paired=scored[scored.matched_control_down_rate.notna()]
    print("Personal confirmations",int(personal.confirmed.sum()),"| scorable",len(scored),
          "| DOWN",int(scored.signal_return.lt(0).sum()),"| no setup",int((~personal.confirmed).sum()))
    if len(paired):
        print(f"Same elapsed session on {len(paired)} matched signals: personal {paired.signal_return.lt(0).mean():.1%} DOWN, controls {paired.matched_control_down_rate.mean():.1%} DOWN.")
if review_code.strip():
    compare_personal(review_code.strip().upper())
